# CSC 4792 Data Mining and Warehousing

## Kabwe Municipal Council Dataset

### Project Information

- **Course:** CSC 4792 Data Mining and Warehousing
- **Academic Year:** 2025/26
- **Council:** Kabwe Municipal Council
- **Country:** Zambia
- **Official Website:** https://www.kabwecouncil.gov.zm

### Objective

The objective of this project is to collect, extract, clean,
transform, and curate information associated with Kabwe Municipal
Council from its official digital sources.

The resulting datasets will be stored as pipe-delimited CSV files
and published on Kaggle.

## 1. Libraries

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
from urllib.parse import urljoin
import pymupdf

In [4]:
print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Data Source

In [5]:
BASE_URL = "https://www.kabwecouncil.gov.zm"

print(BASE_URL)

https://www.kabwecouncil.gov.zm


In [6]:
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

response = requests.get(
    BASE_URL,
    timeout=30,
    verify=False
)

print("Status code:", response.status_code)

Status code: 200


In [7]:
soup = BeautifulSoup(response.text, "html.parser")

In [8]:
links = []

for link in soup.find_all("a", href=True):
    text = link.get_text(" ", strip=True)
    url = urljoin(BASE_URL, link["href"])

    links.append({
        "text": text,
        "url": url
    })

links_df = pd.DataFrame(links)

links_df.head(20)

,text,url
0,,https://www.kabwecouncil.gov.zm#
1,Home,https://www.kabwecouncil.gov.zm/
2,About,https://www.kabwecouncil.gov.zm#
3,About Us,https://www.kabwecouncil.gov.zm/?page_id=2601
4,Mandate,https://www.kabwecouncil.gov.zm/?page_id=169
5,Who we are,https://www.kabwecouncil.gov.zm/?page_id=118
6,Departments,https://www.kabwecouncil.gov.zm/?page_id=770
7,office of the the town clerk,https://www.kabwecouncil.gov.zm/?page_id=2634
8,Dept of Human Resource and Administration,https://www.kabwecouncil.gov.zm/?page_id=2637
9,Dept of Health,https://www.kabwecouncil.gov.zm/?page_id=2640


In [9]:
print("Number of links:", len(links_df))

Number of links: 122


In [10]:
keywords = [
    "cdf",
    "project",
    "budget",
    "financial",
    "report",
    "development",
    "idp",
    "ward",
    "revenue",
    "procurement",
    "minutes"
]

pattern = "|".join(keywords)

relevant_links = links_df[
    links_df["text"].str.contains(
        pattern,
        case=False,
        na=False
    )
]

relevant_links

,text,url
27,CDF,https://www.kabwecouncil.gov.zm/?page_id=2542
28,CDF GUIDLINES,https://www.kabwecouncil.gov.zm/?page_id=2579
29,CDF branding guidelines,https://www.kabwecouncil.gov.zm/?page_id=3674
66,CDF,https://www.kabwecouncil.gov.zm/?page_id=2542
67,CDF GUIDLINES,https://www.kabwecouncil.gov.zm/?page_id=2579
68,CDF branding guidelines,https://www.kabwecouncil.gov.zm/?page_id=3674
91,CDF Skills Bursaries Applicants,https://www.katetecouncil.gov.zm/wp-content/up...
108,CDF PROJECT MONITORING BY KABWE MUNICIPAL COUN...,https://www.kabwecouncil.gov.zm/?p=4386
115,Ministry of Local Government and Rural Develop...,https://www.mlgrd.gov.zm/


In [11]:
cdf_columns = [
    "project_id",
    "year",
    "constituency",
    "project_name",
    "project_description",
    "project_type",
    "ward",
    "project_site",
    "sector",
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount",
    "status",
    "source_url"
]

cdf_projects = pd.DataFrame(columns=cdf_columns)

cdf_projects

,project_id,year,constituency,project_name,project_description,project_type,ward,project_site,sector,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url


In [13]:
import requests

pdf_url = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2024/11/"
    "2024-Bwacha-community-projects-Recieved.pdf"
)

pdf_response = requests.get(
    pdf_url,
    timeout=60,
    verify=False
)

print("Status code:", pdf_response.status_code)
print("Content-Type:", pdf_response.headers.get("Content-Type"))
print("File size:", len(pdf_response.content), "bytes")
print("First 20 bytes:", pdf_response.content[:20])

Status code: 200
Content-Type: application/pdf
File size: 93259 bytes
First 20 bytes: b'%PDF-1.5\r\n%\xb5\xb5\xb5\xb5\r\n1 0'


In [14]:
pdf_path = "../data/raw/2024_bwacha_cdf_projects.pdf"

with open(pdf_path, "wb") as file:
    file.write(pdf_response.content)

print("PDF downloaded successfully.")

PDF downloaded successfully.


In [16]:
import requests
import pdfplumber
import pandas as pd

pdf_url = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2024/11/"
    "2024-Bwacha-community-projects-Recieved.pdf"
)

response = requests.get(
    pdf_url,
    timeout=60,
    verify=False
)

print("Status code:", response.status_code)
print("File size:", len(response.content), "bytes")
print("Content-Type:", response.headers.get("Content-Type"))
print("First 20 bytes:", response.content[:20])

Status code: 200
File size: 93259 bytes
Content-Type: application/pdf
First 20 bytes: b'%PDF-1.5\r\n%\xb5\xb5\xb5\xb5\r\n1 0'


In [17]:
pdf_path = "../data/raw/2024_bwacha_cdf_projects.pdf"

with open(pdf_path, "wb") as file:
    file.write(response.content)

print("PDF saved to:", pdf_path)

PDF saved to: ../data/raw/2024_bwacha_cdf_projects.pdf


In [18]:
with pdfplumber.open(pdf_path) as pdf:
    print("Number of pages:", len(pdf.pages))

    for page_number, page in enumerate(pdf.pages):
        tables = page.extract_tables()

        print(
            f"Page {page_number + 1}: "
            f"{len(tables)} table(s) found"
        )

Number of pages: 5
Page 1: 1 table(s) found
Page 2: 1 table(s) found
Page 3: 1 table(s) found
Page 4: 1 table(s) found
Page 5: 1 table(s) found


In [19]:
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()

    table = tables[0]

    for row in table[:10]:
        print(row)

['2024 CDF COMMUNITY PROJECTS SUBMISSION - BWACHA CONSTITUENCY\nKABWE MUNICIPAL COUNCIL', None, None, None, None, None, None, None, None, None]
['No.', 'Project Name', 'Project Description', 'Ward', 'Project\nSite/Location', 'Application\nAmount', "Engineers'\nEstimates", 'Approved\nAmount', 'Contract\nAmount', 'Status']
['Education', None, None, None, None, None, None, None, None, None]
['1', 'Construction of 1X3 Classroom\nBlock and Teachers Houses in\nKangomba ward', 'Construction of 1X3\nClassroom Block and\nTeachers Houses in\nKangomba ward - Kalima zone', 'Kangomba', 'Kangomba\nward', '', '', '', '', '']
['2', 'Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool', 'Construction of 1X4\nClassroom Block at Kagomba\nPrimary School', 'Kangomba', 'Kangomba\nPrimary\nSchool', '', '', '', '', '']
['3', 'Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool', 'Construction of 1X4\nClassroom Block at Kagomba\nPrimary School', 'Kangomba', 'Kangomba\nPrimary\nSchool', 

In [20]:
df = pd.DataFrame(
    table[1:],
    columns=table[0]
)

df.head()

,2024 CDF COMMUNITY PROJECTS SUBMISSION - BWACHA CONSTITUENCY\nKABWE MUNICIPAL COUNCIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,No.,Project Name,Project Description,Ward,Project\nSite/Location,Application\nAmount,Engineers'\nEstimates,Approved\nAmount,Contract\nAmount,Status
1,Education,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Construction of 1X3 Classroom\nBlock and Teach...,Construction of 1X3\nClassroom Block and\nTeac...,Kangomba,Kangomba\nward,,,,,
3,2,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
4,3,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,


In [21]:
print(df.shape)
print(df.columns.tolist())

(8, 10)
['2024 CDF COMMUNITY PROJECTS SUBMISSION - BWACHA CONSTITUENCY\nKABWE MUNICIPAL COUNCIL', nan, nan, nan, nan, nan, nan, nan, nan, nan]


In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 10 columns):
 #   Column                                                                                Non-Null Count  Dtype
---  ------                                                                                --------------  -----
 0   2024 CDF COMMUNITY PROJECTS SUBMISSION - BWACHA CONSTITUENCY
KABWE MUNICIPAL COUNCIL  8 non-null      str  
 1   nan                                                                                   7 non-null      str  
 2   nan                                                                                   7 non-null      str  
 3   nan                                                                                   7 non-null      str  
 4   nan                                                                                   7 non-null      str  
 5   nan                                                                                   7 non-null      str  
 6   n

In [23]:
for row in table[:10]:
    print(row)

['2024 CDF COMMUNITY PROJECTS SUBMISSION - BWACHA CONSTITUENCY\nKABWE MUNICIPAL COUNCIL', None, None, None, None, None, None, None, None, None]
['No.', 'Project Name', 'Project Description', 'Ward', 'Project\nSite/Location', 'Application\nAmount', "Engineers'\nEstimates", 'Approved\nAmount', 'Contract\nAmount', 'Status']
['Education', None, None, None, None, None, None, None, None, None]
['1', 'Construction of 1X3 Classroom\nBlock and Teachers Houses in\nKangomba ward', 'Construction of 1X3\nClassroom Block and\nTeachers Houses in\nKangomba ward - Kalima zone', 'Kangomba', 'Kangomba\nward', '', '', '', '', '']
['2', 'Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool', 'Construction of 1X4\nClassroom Block at Kagomba\nPrimary School', 'Kangomba', 'Kangomba\nPrimary\nSchool', '', '', '', '', '']
['3', 'Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool', 'Construction of 1X4\nClassroom Block at Kagomba\nPrimary School', 'Kangomba', 'Kangomba\nPrimary\nSchool', 

In [24]:
all_rows = []

with pdfplumber.open(pdf_path) as pdf:
    for page_number, page in enumerate(pdf.pages, start=1):
        tables = page.extract_tables()

        for table in tables:
            for row in table:
                if row:
                    all_rows.append(row)

print("Total rows extracted:", len(all_rows))

Total rows extracted: 47


In [25]:
project_rows = []

for row in all_rows:
    if row[0] is not None and str(row[0]).strip().isdigit():
        project_rows.append(row)

print("Number of project rows:", len(project_rows))

Number of project rows: 36


In [26]:
for row in project_rows[:5]:
    print(row)

['1', 'Construction of 1X3 Classroom\nBlock and Teachers Houses in\nKangomba ward', 'Construction of 1X3\nClassroom Block and\nTeachers Houses in\nKangomba ward - Kalima zone', 'Kangomba', 'Kangomba\nward', '', '', '', '', '']
['2', 'Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool', 'Construction of 1X4\nClassroom Block at Kagomba\nPrimary School', 'Kangomba', 'Kangomba\nPrimary\nSchool', '', '', '', '', '']
['3', 'Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool', 'Construction of 1X4\nClassroom Block at Kagomba\nPrimary School', 'Kangomba', 'Kangomba\nPrimary\nSchool', '', '', '', '', '']
['4', 'Construction of 1X3 Classroom\nBlock at Mine Primary School', 'Construction of 1X3\nClassroom Block at Mine\nPrimary School -\nMutwewansofu zone', 'Kangomba', 'Mine Primary\nSchool', '', '', '', '', '']
['5', 'Repairing of the Mono-Pump,\nConstruction of Toilets for Pre-\nSchool Pupils, Procurement of\nDesks at Mary Chidgey\nCommunity Primary School', 'Repairing

In [27]:
columns = [
    "project_number",
    "project_name",
    "project_description",
    "ward",
    "project_site",
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount",
    "status"
]

df = pd.DataFrame(project_rows, columns=columns)

df.head()

,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of 1X3 Classroom\nBlock and Teach...,Construction of 1X3\nClassroom Block and\nTeac...,Kangomba,Kangomba\nward,,,,,
1,2,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
2,3,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
3,4,Construction of 1X3 Classroom\nBlock at Mine P...,Construction of 1X3\nClassroom Block at Mine\n...,Kangomba,Mine Primary\nSchool,,,,,
4,5,"Repairing of the Mono-Pump,\nConstruction of T...","Repairing of the Mono-Pump,\nConstruction of T...",Kangomba,Mary Chidgey\nCommunity\nSchool,,,,,


In [28]:
print(df.shape)
print(df.columns.tolist())

(36, 10)
['project_number', 'project_name', 'project_description', 'ward', 'project_site', 'application_amount', 'engineers_estimate', 'approved_amount', 'contract_amount', 'status']


In [29]:
for column in df.columns:
    df[column] = (
        df[column]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

df.head()

,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of 1X3 Classroom Block and Teache...,Construction of 1X3 Classroom Block and Teache...,Kangomba,Kangomba ward,,,,,
1,2,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,Kangomba,Kangomba Primary School,,,,,
2,3,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,Kangomba,Kangomba Primary School,,,,,
3,4,Construction of 1X3 Classroom Block at Mine Pr...,Construction of 1X3 Classroom Block at Mine Pr...,Kangomba,Mine Primary School,,,,,
4,5,"Repairing of the Mono-Pump, Construction of To...","Repairing of the Mono-Pump, Construction of To...",Kangomba,Mary Chidgey Community School,,,,,


In [30]:
print(df.loc[0, "project_name"])
print(df.loc[0, "project_description"])

Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward
Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - Kalima zone


In [31]:
df.insert(1, "year", 2024)
df.insert(2, "constituency", "Bwacha")

In [32]:
df.head()

,project_number,year,constituency,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teache...,Construction of 1X3 Classroom Block and Teache...,Kangomba,Kangomba ward,,,,,
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,Kangomba,Kangomba Primary School,,,,,
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,Kangomba,Kangomba Primary School,,,,,
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Pr...,Construction of 1X3 Classroom Block at Mine Pr...,Kangomba,Mine Primary School,,,,,
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of To...","Repairing of the Mono-Pump, Construction of To...",Kangomba,Mary Chidgey Community School,,,,,


In [33]:
df["source_url"] = pdf_url
df["source_document"] = "2024 Bwacha CDF Community Projects Submission"

In [34]:
pd.set_option("display.max_colwidth", 100)

df.head(10)

,project_number,year,constituency,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - Kalima zone,Kangomba,Kangomba ward,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,Kangomba,Mine Primary School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...",Kangomba,Mary Chidgey Community School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
5,6,2024,Bwacha,Construction of a Primary School in Kawama ward,Construction of a Primary School in Kawama ward,Kawama,Kawama ward,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
6,7,2024,Bwacha,Construction of 10 Teachers Houses at Chitakata Community School,Construction of 10 Teachers Houses at Chitakata Community School,Muwowo East,Chitakata Commnuity School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
7,8,2024,Bwacha,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary School,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary School,Muwowo East,Mukobeko Correctional Day Secondary School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
8,9,2024,Bwacha,Construction of Youth Resource Centre,Construction of Youth Resource Centre,Bwacha,Bwacha ward,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
9,10,2024,Bwacha,Construction of a School Hall at Rapheal Kombe Secondary School,Construction of a School Hall at Rapheal Kombe Secondary School,Chimaniman i,Rapheal Kombe Secondary School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission


In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   project_number       36 non-null     str  
 1   year                 36 non-null     int64
 2   constituency         36 non-null     str  
 3   project_name         36 non-null     str  
 4   project_description  36 non-null     str  
 5   ward                 36 non-null     str  
 6   project_site         36 non-null     str  
 7   application_amount   36 non-null     str  
 8   engineers_estimate   36 non-null     str  
 9   approved_amount      36 non-null     str  
 10  contract_amount      36 non-null     str  
 11  status               36 non-null     str  
 12  source_url           36 non-null     str  
 13  source_document      36 non-null     str  
dtypes: int64(1), str(13)
memory usage: 4.1 KB


In [36]:
df["ward"].value_counts()

ward
Kangomba        8
Kawama          8
Chimaniman i    6
Bwacha          4
Muwowo East     2
Munyama         2
Chililalila     2
Chinyama        2
Ngungu          2
Name: count, dtype: int64

In [37]:
df["status"].value_counts(dropna=False)

status
    36
Name: count, dtype: int64

In [38]:
financial_columns = [
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount"
]

for column in financial_columns:
    print(f"\n--- {column} ---")
    print(df[column].unique()[:20])


--- application_amount ---
<StringArray>
['']
Length: 1, dtype: str

--- engineers_estimate ---
<StringArray>
['']
Length: 1, dtype: str

--- approved_amount ---
<StringArray>
['']
Length: 1, dtype: str

--- contract_amount ---
<StringArray>
['']
Length: 1, dtype: str


In [39]:
def clean_amount(value):
    value = str(value).strip()

    # Treat blank values as missing
    if value == "" or value.lower() in ["nan", "none", "n/a", "na", "-"]:
        return pd.NA

    # Remove currency symbols, commas and other non-numeric characters
    value = re.sub(r"[^0-9.\-]", "", value)

    if value == "":
        return pd.NA

    return float(value)

In [40]:
for column in financial_columns:
    df[column] = df[column].apply(clean_amount)

In [41]:
df[financial_columns].head(10)

,application_amount,engineers_estimate,approved_amount,contract_amount
0,<NA>,<NA>,<NA>,<NA>
1,<NA>,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,<NA>
3,<NA>,<NA>,<NA>,<NA>
4,<NA>,<NA>,<NA>,<NA>
5,<NA>,<NA>,<NA>,<NA>
6,<NA>,<NA>,<NA>,<NA>
7,<NA>,<NA>,<NA>,<NA>
8,<NA>,<NA>,<NA>,<NA>
9,<NA>,<NA>,<NA>,<NA>


In [42]:
df[financial_columns].head(10)

,application_amount,engineers_estimate,approved_amount,contract_amount
0,<NA>,<NA>,<NA>,<NA>
1,<NA>,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,<NA>
3,<NA>,<NA>,<NA>,<NA>
4,<NA>,<NA>,<NA>,<NA>
5,<NA>,<NA>,<NA>,<NA>
6,<NA>,<NA>,<NA>,<NA>
7,<NA>,<NA>,<NA>,<NA>
8,<NA>,<NA>,<NA>,<NA>
9,<NA>,<NA>,<NA>,<NA>


In [43]:
df[financial_columns].dtypes

application_amount    object
engineers_estimate    object
approved_amount       object
contract_amount       object
dtype: object

In [44]:
df[financial_columns].isna().sum()

application_amount    36
engineers_estimate    36
approved_amount       36
contract_amount       36
dtype: int64

In [45]:
text_columns = [
    "project_name",
    "project_description",
    "ward",
    "project_site",
    "status"
]

for column in text_columns:
    df[column] = (
        df[column]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

In [46]:
df.head(10)

,project_number,year,constituency,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - Kalima zone,Kangomba,Kangomba ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,Kangomba,Mine Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...",Kangomba,Mary Chidgey Community School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
5,6,2024,Bwacha,Construction of a Primary School in Kawama ward,Construction of a Primary School in Kawama ward,Kawama,Kawama ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
6,7,2024,Bwacha,Construction of 10 Teachers Houses at Chitakata Community School,Construction of 10 Teachers Houses at Chitakata Community School,Muwowo East,Chitakata Commnuity School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
7,8,2024,Bwacha,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary School,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary School,Muwowo East,Mukobeko Correctional Day Secondary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
8,9,2024,Bwacha,Construction of Youth Resource Centre,Construction of Youth Resource Centre,Bwacha,Bwacha ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
9,10,2024,Bwacha,Construction of a School Hall at Rapheal Kombe Secondary School,Construction of a School Hall at Rapheal Kombe Secondary School,Chimaniman i,Rapheal Kombe Secondary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission


In [47]:
duplicates = df.duplicated(
    subset=[
        "project_name",
        "ward",
        "project_site"
    ]
)

print("Duplicate rows:", duplicates.sum())

Duplicate rows: 1


In [48]:
print(df["project_number"].unique())

<StringArray>
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12']
Length: 12, dtype: str


In [49]:
print("Total projects:", len(df))

Total projects: 36


In [50]:
processed_path = "../data/processed/kabwe_2024_bwacha_cdf_projects.csv"

df.to_csv(
    processed_path,
    sep="|",
    index=False
)

print("Saved:", processed_path)

Saved: ../data/processed/kabwe_2024_bwacha_cdf_projects.csv


In [51]:
test_df = pd.read_csv(
    processed_path,
    sep="|"
)

print("Rows:", len(test_df))
print("Columns:", len(test_df.columns))

test_df.head()

Rows: 36
Columns: 14


,project_number,year,constituency,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - Kalima zone,Kangomba,Kangomba ward,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,Kangomba,Mine Primary School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...",Kangomba,Mary Chidgey Community School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission


In [53]:
pdf_url_central = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2024/11/"
    "2024-Community-Projects-Kabwe-Central-received.pdf"
)

response_central = requests.get(
    pdf_url_central,
    timeout=60,
    verify=False
)

print("Status code:", response_central.status_code)
print("Content-Type:", response_central.headers.get("Content-Type"))
print("File size:", len(response_central.content), "bytes")
print("First 20 bytes:", response_central.content[:20])

Status code: 200
Content-Type: application/pdf
File size: 96930 bytes
First 20 bytes: b'%PDF-1.5\r\n%\xb5\xb5\xb5\xb5\r\n1 0'


In [54]:
pdf_path_central = (
    "../data/raw/"
    "2024_kabwe_central_cdf_projects.pdf"
)

with open(pdf_path_central, "wb") as file:
    file.write(response_central.content)

print("Saved:", pdf_path_central)

Saved: ../data/raw/2024_kabwe_central_cdf_projects.pdf


In [55]:
with pdfplumber.open(pdf_path_central) as pdf:
    print("Number of pages:", len(pdf.pages))

    for page_number, page in enumerate(pdf.pages, start=1):
        tables = page.extract_tables()

        print(
            f"Page {page_number}: "
            f"{len(tables)} table(s) found"
        )

Number of pages: 2
Page 1: 1 table(s) found
Page 2: 3 table(s) found


In [56]:
with pdfplumber.open(pdf_path_central) as pdf:
    table = pdf.pages[0].extract_tables()[0]

    for row in table[:10]:
        print(row)

['No.', 'Project Name', 'Project Description', 'Type of\nProject', 'Ward', 'Project\nSite/Location', 'Application\nAmount', "Engineers'\nEstimates", 'Approved\nAmount', 'Contract\nAmount', 'Status']
['Education', None, None, None, None, None, None, None, None, None, None]
['1', 'Construction of Secondary\nSchool 1X4 Classroom\nBlock', 'Construction of Secondary\nSchool 1X4 Classroom\nBlock', 'Construction', 'Luangwa', 'Kabwe Trust\nSecondary School', '', '', '', '', '']
['2', 'Construction of 1X3\nClassroom Block', 'Construction of 1X3\nClassroom Block', 'Construction', 'Luangwa', 'Kabwe Central\nHospital Special\nSchool Community', '', '', '', '', '']
['3', 'Construction of 1X2\nClassroom Block', 'Construction of 1X2\nClassroom Block', 'Construction', 'Luangwa', 'Kabwe Trust Primary\nSchool', '', '', '', '', '']
['4', 'Construction of 1X3\nClassroom Block', 'Construction of 1X3\nClassroom Block', 'Construction', 'Mpima', 'Mpima Dairy Scheme', '', '', '', '', '']
['5', 'Construction of

In [57]:
central_rows = []

with pdfplumber.open(pdf_path_central) as pdf:
    for page in pdf.pages:
        tables = page.extract_tables()

        for table in tables:
            for row in table:
                if row:
                    # Keep only rows whose first column is a project number
                    if (
                        row[0] is not None
                        and str(row[0]).strip().isdigit()
                    ):
                        central_rows.append(row)

print("Kabwe Central project rows:", len(central_rows))

Kabwe Central project rows: 43


In [58]:
central_columns = [
    "project_number",
    "project_name",
    "project_description",
    "project_type",
    "ward",
    "project_site",
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount",
    "status"
]

central_df = pd.DataFrame(
    central_rows,
    columns=central_columns
)

central_df.head()

,project_number,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of Secondary\nSchool 1X4 Classroom\nBlock,Construction of Secondary\nSchool 1X4 Classroom\nBlock,Construction,Luangwa,Kabwe Trust\nSecondary School,,,,,
1,2,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Luangwa,Kabwe Central\nHospital Special\nSchool Community,,,,,
2,3,Construction of 1X2\nClassroom Block,Construction of 1X2\nClassroom Block,Construction,Luangwa,Kabwe Trust Primary\nSchool,,,,,
3,4,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Mpima,Mpima Dairy Scheme,,,,,
4,5,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Mpima,Mpima C,,,,,


In [59]:
print("Rows:", len(central_df))
print("Columns:", len(central_df.columns))
print(central_df.columns.tolist())

Rows: 43
Columns: 11
['project_number', 'project_name', 'project_description', 'project_type', 'ward', 'project_site', 'application_amount', 'engineers_estimate', 'approved_amount', 'contract_amount', 'status']


In [60]:
central_text_columns = [
    "project_name",
    "project_description",
    "project_type",
    "ward",
    "project_site",
    "status"
]

for column in central_text_columns:
    central_df[column] = (
        central_df[column]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

In [61]:
central_df.insert(1, "year", 2024)
central_df.insert(2, "constituency", "Kabwe Central")

In [62]:
central_df["source_url"] = pdf_url_central
central_df["source_document"] = (
    "2024 Kabwe Central CDF Community Projects Submission"
)

In [63]:
central_df.head()

,project_number,year,constituency,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Kabwe Central,Construction of Secondary School 1X4 Classroom Block,Construction of Secondary School 1X4 Classroom Block,Construction,Luangwa,Kabwe Trust Secondary School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Community-Projects-Kabwe-Central...,2024 Kabwe Central CDF Community Projects Submission
1,2,2024,Kabwe Central,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Luangwa,Kabwe Central Hospital Special School Community,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Community-Projects-Kabwe-Central...,2024 Kabwe Central CDF Community Projects Submission
2,3,2024,Kabwe Central,Construction of 1X2 Classroom Block,Construction of 1X2 Classroom Block,Construction,Luangwa,Kabwe Trust Primary School,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Community-Projects-Kabwe-Central...,2024 Kabwe Central CDF Community Projects Submission
3,4,2024,Kabwe Central,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Mpima,Mpima Dairy Scheme,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Community-Projects-Kabwe-Central...,2024 Kabwe Central CDF Community Projects Submission
4,5,2024,Kabwe Central,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Mpima,Mpima C,,,,,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Community-Projects-Kabwe-Central...,2024 Kabwe Central CDF Community Projects Submission


In [64]:
financial_columns = [
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount"
]

for column in financial_columns:
    central_df[column] = central_df[column].apply(
        clean_amount
    )

In [65]:
central_df[financial_columns].head(10)

,application_amount,engineers_estimate,approved_amount,contract_amount
0,<NA>,<NA>,<NA>,<NA>
1,<NA>,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,<NA>
3,<NA>,<NA>,<NA>,<NA>
4,<NA>,<NA>,<NA>,<NA>
5,<NA>,<NA>,<NA>,<NA>
6,<NA>,<NA>,<NA>,<NA>
7,<NA>,<NA>,<NA>,<NA>
8,<NA>,<NA>,<NA>,<NA>
9,<NA>,<NA>,<NA>,<NA>


In [66]:
central_df[financial_columns].isna().sum()

application_amount    43
engineers_estimate    43
approved_amount       43
contract_amount       43
dtype: int64

In [67]:
print("Bwacha columns:")
print(df.columns.tolist())

print("\nKabwe Central columns:")
print(central_df.columns.tolist())

Bwacha columns:
['project_number', 'year', 'constituency', 'project_name', 'project_description', 'ward', 'project_site', 'application_amount', 'engineers_estimate', 'approved_amount', 'contract_amount', 'status', 'source_url', 'source_document']

Kabwe Central columns:
['project_number', 'year', 'constituency', 'project_name', 'project_description', 'project_type', 'ward', 'project_site', 'application_amount', 'engineers_estimate', 'approved_amount', 'contract_amount', 'status', 'source_url', 'source_document']


In [68]:
df["project_type"] = pd.NA

In [69]:
central_columns_final = [
    "project_number",
    "year",
    "constituency",
    "project_name",
    "project_description",
    "project_type",
    "ward",
    "project_site",
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount",
    "status",
    "source_url",
    "source_document"
]

df = df[central_columns_final]
central_df = central_df[central_columns_final]

In [70]:
kabwe_cdf_projects = pd.concat(
    [df, central_df],
    ignore_index=True
)

print(
    "Total Kabwe CDF projects:",
    len(kabwe_cdf_projects)
)

Total Kabwe CDF projects: 79


In [71]:
kabwe_cdf_projects["constituency"].value_counts()

constituency
Kabwe Central    43
Bwacha           36
Name: count, dtype: int64

In [72]:
output_path = (
    "../data/processed/"
    "kabwe_cdf_projects_2024.csv"
)

kabwe_cdf_projects.to_csv(
    output_path,
    sep="|",
    index=False
)

print("Saved:", output_path)

Saved: ../data/processed/kabwe_cdf_projects_2024.csv


In [73]:
kabwe_cdf_projects.info()

<class 'pandas.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   project_number       79 non-null     str   
 1   year                 79 non-null     int64 
 2   constituency         79 non-null     str   
 3   project_name         79 non-null     str   
 4   project_description  79 non-null     str   
 5   project_type         43 non-null     object
 6   ward                 79 non-null     str   
 7   project_site         79 non-null     str   
 8   application_amount   0 non-null      object
 9   engineers_estimate   0 non-null      object
 10  approved_amount      0 non-null      object
 11  contract_amount      0 non-null      object
 12  status               79 non-null     str   
 13  source_url           79 non-null     str   
 14  source_document      79 non-null     str   
dtypes: int64(1), object(5), str(9)
memory usage: 9.4+ KB


In [74]:
print("Rows:", len(kabwe_cdf_projects))
print("Columns:", len(kabwe_cdf_projects.columns))

Rows: 79
Columns: 15


In [75]:
missing = kabwe_cdf_projects.isna().sum()

print(missing)

project_number          0
year                    0
constituency            0
project_name            0
project_description     0
project_type           36
ward                    0
project_site            0
application_amount     79
engineers_estimate     79
approved_amount        79
contract_amount        79
status                  0
source_url              0
source_document         0
dtype: int64


In [76]:
missing_percentage = (
    kabwe_cdf_projects.isna().mean() * 100
).round(2)

print(missing_percentage)

project_number           0.00
year                     0.00
constituency             0.00
project_name             0.00
project_description      0.00
project_type            45.57
ward                     0.00
project_site             0.00
application_amount     100.00
engineers_estimate     100.00
approved_amount        100.00
contract_amount        100.00
status                   0.00
source_url               0.00
source_document          0.00
dtype: float64


In [77]:
empty_values = (
    kabwe_cdf_projects
    .astype(str)
    .apply(lambda column: column.str.strip().eq("").sum())
)

print(empty_values)

project_number          0
year                    0
constituency            0
project_name            0
project_description     0
project_type            0
ward                    0
project_site            1
application_amount      0
engineers_estimate      0
approved_amount         0
contract_amount         0
status                 79
source_url              0
source_document         0
dtype: int64


In [78]:
duplicate_count = kabwe_cdf_projects.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)

Exact duplicate rows: 0


In [79]:
project_duplicates = kabwe_cdf_projects.duplicated(
    subset=[
        "year",
        "constituency",
        "project_name",
        "ward",
        "project_site"
    ],
    keep=False
)

print(
    "Potential duplicate project records:",
    project_duplicates.sum()
)

Potential duplicate project records: 2


In [80]:
kabwe_cdf_projects[
    project_duplicates
].sort_values(
    ["constituency", "project_name"]
)

,project_number,year,constituency,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission


In [81]:
print(
    kabwe_cdf_projects["year"].value_counts()
)

year
2024    79
Name: count, dtype: int64


In [82]:
print(
    kabwe_cdf_projects["constituency"].value_counts()
)

constituency
Kabwe Central    43
Bwacha           36
Name: count, dtype: int64


In [83]:
print(
    kabwe_cdf_projects[
        financial_columns
    ].dtypes
)

application_amount    object
engineers_estimate    object
approved_amount       object
contract_amount       object
dtype: object


In [84]:
kabwe_cdf_projects[
    financial_columns
].describe()

,application_amount,engineers_estimate,approved_amount,contract_amount
count,0,0,0,0
unique,0,0,0,0
top,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN


In [85]:
for column in financial_columns:
    negative_values = (
        kabwe_cdf_projects[column] < 0
    ).sum()

    print(
        column,
        "negative values:",
        negative_values
    )

application_amount negative values: 0
engineers_estimate negative values: 0
approved_amount negative values: 0
contract_amount negative values: 0


In [86]:
print(
    kabwe_cdf_projects[
        "project_name"
    ].head(20).to_string(index=False)
)

                            Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward
                                       Construction of 1X4 Classroom Block at Kagomba Primary School
                                       Construction of 1X4 Classroom Block at Kagomba Primary School
                                          Construction of 1X3 Classroom Block at Mine Primary School
Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks ...
                                                     Construction of a Primary School in Kawama ward
                                    Construction of 10 Teachers Houses at Chitakata Community School
                   Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary School
                                                               Construction of Youth Resource Centre
                                     Construction of a School Hall at Rapheal Kombe Seconda

In [87]:
quality_report = pd.DataFrame({
    "column": kabwe_cdf_projects.columns,
    "data_type": [
        str(dtype)
        for dtype in kabwe_cdf_projects.dtypes
    ],
    "missing_values": [
        kabwe_cdf_projects[column].isna().sum()
        for column in kabwe_cdf_projects.columns
    ],
    "unique_values": [
        kabwe_cdf_projects[column].nunique()
        for column in kabwe_cdf_projects.columns
    ]
})

quality_report

,column,data_type,missing_values,unique_values
0,project_number,str,0,26
1,year,int64,0,1
2,constituency,str,0,2
3,project_name,str,0,75
4,project_description,str,0,75
5,project_type,object,36,8
6,ward,str,0,25
7,project_site,str,0,51
8,application_amount,object,79,0
9,engineers_estimate,object,79,0


In [4]:
import requests

In [7]:
import requests
import urllib3

urllib3.disable_warnings(
    urllib3.exceptions.InsecureRequestWarning
)

pdf_url_2025_central = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2025/08/"
    "Proposed-2025-CDF-projects.pdf"
)

response_2025_central = requests.get(
    pdf_url_2025_central,
    timeout=60,
    verify=False
)

print("Status code:", response_2025_central.status_code)
print("Content-Type:", response_2025_central.headers.get("Content-Type"))
print("File size:", len(response_2025_central.content), "bytes")
print("First 20 bytes:", response_2025_central.content[:20])

Status code: 200
Content-Type: application/pdf
File size: 2017833 bytes
First 20 bytes: b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n11 0 '


In [8]:
pdf_path_2025_central = (
    "../data/raw/"
    "2025_proposed_kabwe_central_cdf_projects.pdf"
)

with open(pdf_path_2025_central, "wb") as file:
    file.write(response_2025_central.content)

print("Saved:", pdf_path_2025_central)

Saved: ../data/raw/2025_proposed_kabwe_central_cdf_projects.pdf


In [9]:
pdf_path_2025_central = (
    "../data/raw/"
    "2025_proposed_kabwe_central_cdf_projects.pdf"
)

with open(pdf_path_2025_central, "wb") as file:
    file.write(response_2025_central.content)

print("Saved:", pdf_path_2025_central)

Saved: ../data/raw/2025_proposed_kabwe_central_cdf_projects.pdf


In [12]:
import pdfplumber

In [14]:
with pdfplumber.open(pdf_path_2025_central) as pdf:
    for page_number, page in enumerate(pdf.pages, start=1):
        text = page.extract_text()

        print(
            f"Page {page_number}:",
            "TEXT FOUND" if text else "NO TEXT"
        )

Page 1: NO TEXT
Page 2: NO TEXT
Page 3: NO TEXT
Page 4: NO TEXT
Page 5: NO TEXT


In [15]:
import fitz

document = fitz.open(pdf_path_2025_central)

for page_number, page in enumerate(document, start=1):
    text = page.get_text("text")

    print(
        f"Page {page_number}:",
        len(text),
        "characters"
    )

Page 1: 0 characters
Page 2: 0 characters
Page 3: 0 characters
Page 4: 0 characters
Page 5: 0 characters


In [17]:
!apt-get update -qq
!apt-get install -y tesseract-ocr
!pip install pytesseract pdf2image

'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'apt-get' is not recognized as an internal or external command,
operable program or batch file.



   -------------------- ------------------- 1/2 [pdf2image]
   ---------------------------------------- 2/2 [pdf2image]



In [18]:
import pytesseract

print(pytesseract.get_tesseract_version())

TesseractNotFoundError: tesseract is not installed or it's not in your PATH. See README file for more information.

In [19]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

print(pytesseract.get_tesseract_version())

5.5.3.20260724


In [20]:
from pdf2image import convert_from_path

print("pdf2image is ready")

pdf2image is ready


In [23]:
from pdf2image import convert_from_path

poppler_path = r"C:\poppler-26.07.0\Library\bin"

pages = convert_from_path(
    pdf_path_2025_central,
    dpi=300,
    poppler_path=poppler_path
)

print("Pages converted:", len(pages))

Pages converted: 5


In [24]:
page_text = pytesseract.image_to_string(
    pages[0],
    config="--psm 6"
)

print(page_text[:5000])

_
CDF 2025 PROPOSED COMMUNITY PROJECT KABWE CENTRAL CONSTITUENCY ©
©
NO | NAME OF COMMUNITY PROJECT APPLIED FORSHORTLUSTED; WARD SECTOR COMMENT O
1 Proposed Construction of a standard Maternity Annex at Mpima 5
Mpima health center
.
— =
Proposed Construction of a 1x4 Classroom block ( CRB) at High ridge Approved =
Lukanga Secondary School i
o
7 Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa =
Kasanda Malombe Secondary School Oo
O
4 | Proposed Construction and installation of 02 Solar powered | Waya Water and Approved
Water reticulated Systems with lockable kiosks in Waya Sanitation
communities
Proposed Construction and installation of 02 Solar powered | Kalonga Water and Approved
Water reticulated Systems at Kamushanga Market Shelter Sanitation
Proposed Construction and installation of 02 Solar powered | Kaputula Education Approved
Water reticulated Systems at C-gate Priamary School
7 | Proposed Construction and installation of 02 Solar powered | Nijanji Education Appro

In [25]:
ocr_path = (
    "../data/raw/"
    "2025_kabwe_central_cdf_ocr.txt"
)

with open(ocr_path, "w", encoding="utf-8") as file:
    file.write(page_text)

print("OCR text saved:", ocr_path)

OCR text saved: ../data/raw/2025_kabwe_central_cdf_ocr.txt


In [26]:
print(page_text[:5000])

_
CDF 2025 PROPOSED COMMUNITY PROJECT KABWE CENTRAL CONSTITUENCY ©
©
NO | NAME OF COMMUNITY PROJECT APPLIED FORSHORTLUSTED; WARD SECTOR COMMENT O
1 Proposed Construction of a standard Maternity Annex at Mpima 5
Mpima health center
.
— =
Proposed Construction of a 1x4 Classroom block ( CRB) at High ridge Approved =
Lukanga Secondary School i
o
7 Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa =
Kasanda Malombe Secondary School Oo
O
4 | Proposed Construction and installation of 02 Solar powered | Waya Water and Approved
Water reticulated Systems with lockable kiosks in Waya Sanitation
communities
Proposed Construction and installation of 02 Solar powered | Kalonga Water and Approved
Water reticulated Systems at Kamushanga Market Shelter Sanitation
Proposed Construction and installation of 02 Solar powered | Kaputula Education Approved
Water reticulated Systems at C-gate Priamary School
7 | Proposed Construction and installation of 02 Solar powered | Nijanji Education Appro

In [27]:
all_ocr_text = []

for page_number, page in enumerate(pages, start=1):
    text = pytesseract.image_to_string(
        page,
        config="--psm 6"
    )

    all_ocr_text.append(text)

    print(f"Page {page_number} OCR complete")

Page 1 OCR complete
Page 2 OCR complete
Page 3 OCR complete
Page 4 OCR complete
Page 5 OCR complete


In [28]:
full_ocr_text = "\n".join(all_ocr_text)

print(full_ocr_text[:10000])

_
CDF 2025 PROPOSED COMMUNITY PROJECT KABWE CENTRAL CONSTITUENCY ©
©
NO | NAME OF COMMUNITY PROJECT APPLIED FORSHORTLUSTED; WARD SECTOR COMMENT O
1 Proposed Construction of a standard Maternity Annex at Mpima 5
Mpima health center
.
— =
Proposed Construction of a 1x4 Classroom block ( CRB) at High ridge Approved =
Lukanga Secondary School i
o
7 Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa =
Kasanda Malombe Secondary School Oo
O
4 | Proposed Construction and installation of 02 Solar powered | Waya Water and Approved
Water reticulated Systems with lockable kiosks in Waya Sanitation
communities
Proposed Construction and installation of 02 Solar powered | Kalonga Water and Approved
Water reticulated Systems at Kamushanga Market Shelter Sanitation
Proposed Construction and installation of 02 Solar powered | Kaputula Education Approved
Water reticulated Systems at C-gate Priamary School
7 | Proposed Construction and installation of 02 Solar powered | Nijanji Education Appro

In [29]:
ocr_path = (
    "../data/raw/"
    "2025_kabwe_central_cdf_ocr.txt"
)

with open(ocr_path, "w", encoding="utf-8") as file:
    file.write(full_ocr_text)

print("Complete OCR saved:", ocr_path)

Complete OCR saved: ../data/raw/2025_kabwe_central_cdf_ocr.txt


In [31]:
import re

In [36]:
project_lines = []

for line in lines:
    line = line.strip()

    if re.match(r"^\d+\s", line):
        project_lines.append(line)

for line in project_lines:
    print(line)

1 Proposed Construction of a standard Maternity Annex at Mpima 5
7 Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa =
4 | Proposed Construction and installation of 02 Solar powered | Waya Water and Approved
7 | Proposed Construction and installation of 02 Solar powered | Nijanji Education Approved
10 | Proposed Construction of an Ablution block at BOCCs Katondo Approved c
11 | Procurement of a Hydraulic Tipper Truck All Transport Approved OD
12 | Proposed extension and rehabilitation of Waya market Waya Commerce and | Approved 140]
13 | Proposed Completion of Nakoli Market shelter Nakoli Commerce and | Approved
14 | Completion of Kabwe General Hospital’s Relative waiting Luangwa Health Approved
15 | Construction of an Ablution Block at Mpima Prison Primary | MPIMA Approved
17 | Additional Roads Funding for roads ALL Transport Approved =
19 | Additional funding for Kasanda Market Justin Commerce and | Approved =
20 | Proposed Construction of a 1x4 Classroom block ( CRB) at

In [37]:
print("Number of numbered rows:", len(project_lines))

Number of numbered rows: 25


In [38]:
for i, line in enumerate(project_lines, start=1):
    print(f"{i}: {line}")

1: 1 Proposed Construction of a standard Maternity Annex at Mpima 5
2: 7 Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa =
3: 4 | Proposed Construction and installation of 02 Solar powered | Waya Water and Approved
4: 7 | Proposed Construction and installation of 02 Solar powered | Nijanji Education Approved
5: 10 | Proposed Construction of an Ablution block at BOCCs Katondo Approved c
6: 11 | Procurement of a Hydraulic Tipper Truck All Transport Approved OD
7: 12 | Proposed extension and rehabilitation of Waya market Waya Commerce and | Approved 140]
8: 13 | Proposed Completion of Nakoli Market shelter Nakoli Commerce and | Approved
9: 14 | Completion of Kabwe General Hospital’s Relative waiting Luangwa Health Approved
10: 15 | Construction of an Ablution Block at Mpima Prison Primary | MPIMA Approved
11: 17 | Additional Roads Funding for roads ALL Transport Approved =
12: 19 | Additional funding for Kasanda Market Justin Commerce and | Approved =
13: 20 | Proposed Cons

In [39]:
ocr_rows_path = "../data/raw/2025_kabwe_central_cdf_project_rows_ocr.txt"

with open(ocr_rows_path, "w", encoding="utf-8") as file:
    for i, line in enumerate(project_lines, start=1):
        file.write(f"{i}: {line}\n")

print("Saved:", ocr_rows_path)

Saved: ../data/raw/2025_kabwe_central_cdf_project_rows_ocr.txt


In [40]:
from pytesseract import Output

ocr_data = pytesseract.image_to_data(
    pages[0],
    config="--psm 6",
    output_type=Output.DATAFRAME
)

ocr_data = ocr_data.dropna(subset=["text"])

ocr_data["text"] = (
    ocr_data["text"]
    .astype(str)
    .str.strip()
)

ocr_data = ocr_data[ocr_data["text"] != ""]

ocr_data = ocr_data.sort_values(
    ["top", "left"]
)

print(
    ocr_data[
        ["top", "left", "width", "text"]
    ].to_string(index=False)
)

 top  left  width            text
  72  3390     55               _
 106  3390     54               ©
 162  1191    295       COMMUNITY
 163   709     85             CDF
 163   808    106            2025
 163   932    242        PROPOSED
 164  1503    194         PROJECT
 165  1713    163           KABWE
 165  1893    206         CENTRAL
 166  2113    344    CONSTITUENCY
 279  3389     56               ©
 330  3390     55               O
 348  2001    172          SECTOR
 349   197     29               |
 353  1667    143            WARD
 353  2589    247         COMMENT
 355   979    185         APPLIED
 355  1192    391 FORSHORTLUSTED;
 356   232    136            NAME
 356   384     60              OF
 356   774    192         PROJECT
 357   110     67              NO
 357   459    299       COMMUNITY
 538  3369     76               5
 581   841    191        standard
 582   745     43              of
 582  1050    216       Maternity
 582  1616    151           Mpima
 583  1279    

In [41]:
project_numbers = []

for line in project_lines:
    match = re.match(r"^(\d+)", line)

    if match:
        project_numbers.append(int(match.group(1)))

print(project_numbers)

[1, 7, 4, 7, 10, 11, 12, 13, 14, 15, 17, 19, 20, 21, 22, 23, 24, 26, 27, 28, 29, 30, 31, 32, 33]


In [42]:
for i, line in enumerate(project_lines, start=1):
    print(f"\nROW {i}")
    print(line)


ROW 1
1 Proposed Construction of a standard Maternity Annex at Mpima 5

ROW 2
7 Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa =

ROW 3
4 | Proposed Construction and installation of 02 Solar powered | Waya Water and Approved

ROW 4
7 | Proposed Construction and installation of 02 Solar powered | Nijanji Education Approved

ROW 5
10 | Proposed Construction of an Ablution block at BOCCs Katondo Approved c

ROW 6
11 | Procurement of a Hydraulic Tipper Truck All Transport Approved OD

ROW 7
12 | Proposed extension and rehabilitation of Waya market Waya Commerce and | Approved 140]

ROW 8
13 | Proposed Completion of Nakoli Market shelter Nakoli Commerce and | Approved

ROW 9
14 | Completion of Kabwe General Hospital’s Relative waiting Luangwa Health Approved

ROW 10
15 | Construction of an Ablution Block at Mpima Prison Primary | MPIMA Approved

ROW 11
17 | Additional Roads Funding for roads ALL Transport Approved =

ROW 12
19 | Additional funding for Kasanda Market Justin 

In [45]:
# Recreate OCR data for page 1
page1_ocr = pytesseract.image_to_data(
    pages[0],
    config="--psm 6",
    output_type=Output.DATAFRAME
)

page1_ocr = page1_ocr.dropna(subset=["text"])

page1_ocr["text"] = (
    page1_ocr["text"]
    .astype(str)
    .str.strip()
)

page1_ocr = page1_ocr[
    page1_ocr["text"] != ""
]

# Find numbers in the NO column
page1_numbers = page1_ocr[
    (page1_ocr["left"] < 210) &
    (page1_ocr["text"].str.match(r"^\d+$", na=False))
]

print(
    page1_numbers[
        ["top", "left", "text"]
    ].to_string(index=False)
)

 top  left text
 587   111    1
1026    76    7
1296   100    4
2177    93    7


In [46]:
page1_ocr = pytesseract.image_to_data(
    pages[0],
    config="--psm 6",
    output_type=Output.DATAFRAME
)

page1_ocr = page1_ocr.dropna(subset=["text"])

page1_ocr["text"] = (
    page1_ocr["text"]
    .astype(str)
    .str.strip()
)

page1_ocr = page1_ocr[
    page1_ocr["text"] != ""
]

page1_numbers = page1_ocr[
    (page1_ocr["left"] < 210) &
    (page1_ocr["text"].str.match(r"^\d+$", na=False))
]

print(
    page1_numbers[
        ["top", "left", "text"]
    ].to_string(index=False)
)

 top  left text
 587   111    1
1026    76    7
1296   100    4
2177    93    7


In [47]:
row1 = page1_ocr[
    (page1_ocr["top"] >= 500) &
    (page1_ocr["top"] <= 750)
].copy()

print(
    row1[
        ["top", "left", "text"]
    ].sort_values(["top", "left"]).to_string(index=False)
)

 top  left         text
 538  3369            5
 581   841     standard
 582   745           of
 582  1050    Maternity
 582  1616        Mpima
 583  1279        Annex
 584   231     Proposed
 585   450 Construction
 586  1430           at
 587   111            1
 592   802            a
 655   396       health
 656   230        Mpima
 660   546       center
 684  3365            .
 745  3370            =


In [48]:
row2 = page1_ocr[
    (page1_ocr["top"] >= 780) &
    (page1_ocr["top"] <= 1000)
].copy()

print(
    row2[
        ["top", "left", "text"]
    ].sort_values(["top", "left"]).to_string(index=False)
)

 top  left         text
 781  1805            —
 799  3390            =
 815  2310     Approved
 817   738           of
 817   927    Classroom
 818  1172        block
 819   839          1x4
 819  1302            (
 819  1329         CRB)
 819  1615         High
 819  1727        ridge
 820   229     Proposed
 821   446 Construction
 826  1444           at
 828   797            a
 889   655       School
 891   416    Secondary
 893   228      Lukanga
 903  3370            i
 960  3390            o


In [49]:
page1_ocr.sort_values(
    ["top", "left"]
)[
    ["top", "left", "text"]
].to_string(index=False)

' top  left            text\n  72  3390               _\n 106  3390               ©\n 162  1191       COMMUNITY\n 163   709             CDF\n 163   808            2025\n 163   932        PROPOSED\n 164  1503         PROJECT\n 165  1713           KABWE\n 165  1893         CENTRAL\n 166  2113    CONSTITUENCY\n 279  3389               ©\n 330  3390               O\n 348  2001          SECTOR\n 349   197               |\n 353  1667            WARD\n 353  2589         COMMENT\n 355   979         APPLIED\n 355  1192 FORSHORTLUSTED;\n 356   232            NAME\n 356   384              OF\n 356   774         PROJECT\n 357   110              NO\n 357   459       COMMUNITY\n 538  3369               5\n 581   841        standard\n 582   745              of\n 582  1050       Maternity\n 582  1616           Mpima\n 583  1279           Annex\n 584   231        Proposed\n 585   450    Construction\n 586  1430              at\n 587   111               1\n 592   802               a\n 655   396         

In [50]:
working_df = pd.DataFrame({
    "ocr_row": range(1, len(project_lines) + 1),
    "ocr_text": project_lines
})

working_df

NameError: name 'pd' is not defined

In [51]:
import pandas as pd
import re

In [52]:
working_df = pd.DataFrame({
    "ocr_row": range(1, len(project_lines) + 1),
    "ocr_text": project_lines
})

working_df

,ocr_row,ocr_text
0,1,1 Proposed Construction of a standard Maternit...
1,2,7 Proposed Construction of a 1x3 Classroom blo...
2,3,4 | Proposed Construction and installation of ...
3,4,7 | Proposed Construction and installation of ...
4,5,10 | Proposed Construction of an Ablution bloc...
5,6,11 | Procurement of a Hydraulic Tipper Truck A...
6,7,12 | Proposed extension and rehabilitation of ...
7,8,13 | Proposed Completion of Nakoli Market shel...
8,9,14 | Completion of Kabwe General Hospital’s Re...
9,10,15 | Construction of an Ablution Block at Mpim...


In [53]:
working_df[
    ["ocr_row", "no_ocr", "ocr_text"]
].to_string(index=False)

KeyError: "['no_ocr'] not in index"

In [54]:
working_df["no_ocr"] = working_df["ocr_text"].str.extract(
    r"^(\d+)"
)[0]

print(working_df.columns.tolist())

['ocr_row', 'ocr_text', 'no_ocr']


In [55]:
print(
    working_df[
        ["ocr_row", "no_ocr", "ocr_text"]
    ].to_string(index=False)
)

 ocr_row no_ocr                                                                                                              ocr_text
       1      1                                                      1 Proposed Construction of a standard Maternity Annex at Mpima 5
       2      7                                                    7 Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa =
       3      4                              4 | Proposed Construction and installation of 02 Solar powered | Waya Water and Approved
       4      7                           7 | Proposed Construction and installation of 02 Solar powered | Nijanji Education Approved
       5     10                                           10 | Proposed Construction of an Ablution block at BOCCs Katondo Approved c
       6     11                                                11 | Procurement of a Hydraulic Tipper Truck All Transport Approved OD
       7     12                           12 | Proposed extens

In [56]:
working_df["project_text"] = (
    working_df["ocr_text"]
    .str.replace(r"^\d+\s*", "", regex=True)
    .str.strip()
)

working_df[
    ["ocr_row", "no_ocr", "project_text"]
]

,ocr_row,no_ocr,project_text
0,1,1,Proposed Construction of a standard Maternity ...
1,2,7,Proposed Construction of a 1x3 Classroom block...
2,3,4,| Proposed Construction and installation of 02...
3,4,7,| Proposed Construction and installation of 02...
4,5,10,| Proposed Construction of an Ablution block a...
5,6,11,| Procurement of a Hydraulic Tipper Truck All ...
6,7,12,| Proposed extension and rehabilitation of Way...
7,8,13,| Proposed Completion of Nakoli Market shelter...
8,9,14,| Completion of Kabwe General Hospital’s Relat...
9,10,15,| Construction of an Ablution Block at Mpima P...


In [57]:
working_df["project_text"] = (
    working_df["project_text"]
    .str.replace(r"\s*[=—]+$", "", regex=True)
    .str.strip()
)

print(
    working_df[
        ["ocr_row", "no_ocr", "project_text"]
    ].to_string(index=False)
)

 ocr_row no_ocr                                                                                                       project_text
       1      1                                                     Proposed Construction of a standard Maternity Annex at Mpima 5
       2      7                                                     Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa
       3      4                             | Proposed Construction and installation of 02 Solar powered | Waya Water and Approved
       4      7                          | Proposed Construction and installation of 02 Solar powered | Nijanji Education Approved
       5     10                                           | Proposed Construction of an Ablution block at BOCCs Katondo Approved c
       6     11                                                | Procurement of a Hydraulic Tipper Truck All Transport Approved OD
       7     12                           | Proposed extension and rehabilitation o

In [58]:
working_df["project_text"] = (
    working_df["project_text"]
    .str.replace(r"\s*[=—]+\s*$", "", regex=True)
    .str.strip()
)

working_df[
    ["ocr_row", "no_ocr", "project_text"]
]

,ocr_row,no_ocr,project_text
0,1,1,Proposed Construction of a standard Maternity ...
1,2,7,Proposed Construction of a 1x3 Classroom block...
2,3,4,| Proposed Construction and installation of 02...
3,4,7,| Proposed Construction and installation of 02...
4,5,10,| Proposed Construction of an Ablution block a...
5,6,11,| Procurement of a Hydraulic Tipper Truck All ...
6,7,12,| Proposed extension and rehabilitation of Way...
7,8,13,| Proposed Completion of Nakoli Market shelter...
8,9,14,| Completion of Kabwe General Hospital’s Relat...
9,10,15,| Construction of an Ablution Block at Mpima P...


In [59]:
# Show OCR text that appears in the Ward, Sector and Comment areas
page1_columns = page1_ocr[
    page1_ocr["left"] >= 1550
].copy()

print(
    page1_columns[
        ["top", "left", "text"]
    ].sort_values(["top", "left"]).to_string(index=False)
)

 top  left         text
  72  3390            _
 106  3390            ©
 165  1713        KABWE
 165  1893      CENTRAL
 166  2113 CONSTITUENCY
 279  3389            ©
 330  3390            O
 348  2001       SECTOR
 353  1667         WARD
 353  2589      COMMENT
 538  3369            5
 582  1616        Mpima
 684  3365            .
 745  3370            =
 781  1805            —
 799  3390            =
 815  2310     Approved
 819  1615         High
 819  1727        ridge
 903  3370            i
 960  3390            o
1017  3389            =
1052  1614       Chirwa
1129  3389           Oo
1183  3390            O
1287  1571            |
1290  2067          and
1292  2312     Approved
1293  1613         Waya
1293  1915        Water
1359  1915   Sanitation
1673  1564            |
1677  1615      Kalonga
1681  2070          and
1681  2316     Approved
1682  1917        Water
1755  1917   Sanitation
1920  1565            |
1922  1917    Education
1925  1615     Kaputula
1926  2318     A

In [60]:
# Define the approximate column boundaries
WARD_MIN = 1550
WARD_MAX = 1850

SECTOR_MIN = 1850
SECTOR_MAX = 2250

COMMENT_MIN = 2250
COMMENT_MAX = 3000

# Extract text from each column
ward_data = page1_ocr[
    (page1_ocr["left"] >= WARD_MIN) &
    (page1_ocr["left"] < WARD_MAX)
].copy()

sector_data = page1_ocr[
    (page1_ocr["left"] >= SECTOR_MIN) &
    (page1_ocr["left"] < SECTOR_MAX)
].copy()

comment_data = page1_ocr[
    (page1_ocr["left"] >= COMMENT_MIN) &
    (page1_ocr["left"] < COMMENT_MAX)
].copy()

print("WARD:")
print(
    ward_data[
        ["top", "left", "text"]
    ].sort_values(["top", "left"]).to_string(index=False)
)

print("\nSECTOR:")
print(
    sector_data[
        ["top", "left", "text"]
    ].sort_values(["top", "left"]).to_string(index=False)
)

print("\nCOMMENT:")
print(
    comment_data[
        ["top", "left", "text"]
    ].sort_values(["top", "left"]).to_string(index=False)
)

WARD:
 top  left     text
 165  1713    KABWE
 353  1667     WARD
 582  1616    Mpima
 781  1805        —
 819  1615     High
 819  1727    ridge
1052  1614   Chirwa
1287  1571        |
1293  1613     Waya
1673  1564        |
1677  1615  Kalonga
1920  1565        |
1925  1615 Kaputula
2169  1564        |
2175  1615  Nijanji

SECTOR:
 top  left         text
 165  1893      CENTRAL
 166  2113 CONSTITUENCY
 348  2001       SECTOR
1290  2067          and
1293  1915        Water
1359  1915   Sanitation
1681  2070          and
1682  1917        Water
1755  1917   Sanitation
1922  1917    Education
2166  1919    Education

COMMENT:
 top  left     text
 353  2589  COMMENT
 815  2310 Approved
1292  2312 Approved
1681  2316 Approved
1926  2318 Approved
2169  2316 Approved


In [61]:
def get_row_text(data, start_top, end_top, min_left=0, max_left=3000):
    row = data[
        (data["top"] >= start_top) &
        (data["top"] < end_top) &
        (data["left"] >= min_left) &
        (data["left"] < max_left)
    ].copy()

    row = row.sort_values(["top", "left"])

    return " ".join(row["text"].tolist())


# Test with the first project
row1_text = get_row_text(
    page1_ocr,
    500,
    780,
    50,
    3000
)

print(row1_text)

standard of Maternity Mpima Annex Proposed Construction at 1 a health Mpima center


In [62]:
# Display the first project's OCR words with their coordinates
row1 = page1_ocr[
    (page1_ocr["top"] >= 530) &
    (page1_ocr["top"] <= 700)
].copy()

row1 = row1.sort_values(["top", "left"])

print(
    row1[
        ["top", "left", "text"]
    ].to_string(index=False)
)

 top  left         text
 538  3369            5
 581   841     standard
 582   745           of
 582  1050    Maternity
 582  1616        Mpima
 583  1279        Annex
 584   231     Proposed
 585   450 Construction
 586  1430           at
 587   111            1
 592   802            a
 655   396       health
 656   230        Mpima
 660   546       center
 684  3365            .


In [63]:
# Page 1
page1 = pages[0]

# Row 1 vertical range
Y1 = 530
Y2 = 710

# Approximate table column boundaries
columns = {
    "NO": (0, 210),
    "PROJECT": (210, 1550),
    "WARD": (1550, 1850),
    "SECTOR": (1850, 2250),
    "COMMENT": (2250, 3000)
}

for column_name, (x1, x2) in columns.items():
    crop = page1.crop((x1, Y1, x2, Y2))

    text = pytesseract.image_to_string(
        crop,
        config="--psm 6"
    )

    print(f"\n--- {column_name} ---")
    print(text.strip())


--- NO ---
P|

--- PROJECT ---
Proposed Construction of a standard Maternity Annex at
Mpima health center

--- WARD ---
a
| Mpima

--- SECTOR ---
| Health

--- COMMENT ---
| Approved


In [64]:
# Row 2 vertical range
Y1 = 780
Y2 = 970

for column_name, (x1, x2) in columns.items():
    crop = page1.crop((x1, Y1, x2, Y2))

    text = pytesseract.image_to_string(
        crop,
        config="--psm 6"
    )

    print(f"\n--- {column_name} ---")
    print(text.strip())


--- NO ---
P |

--- PROJECT ---
Proposed Construction of a 1x4 Classroom block ( CRB) at
Lukanga Secondary School

--- WARD ---
High ridge

--- SECTOR ---
Education

--- COMMENT ---
Approved


In [65]:
# Row 3 vertical range
Y1 = 1170
Y2 = 1460

for column_name, (x1, x2) in columns.items():
    crop = page1.crop((x1, Y1, x2, Y2))

    text = pytesseract.image_to_string(
        crop,
        config="--psm 6"
    )

    print(f"\n--- {column_name} ---")
    print(text.strip())


--- NO ---
7

--- PROJECT ---
Proposed Construction and installation of 02 Solar powered
Water reticulated Systems with lockable kiosks in Waya

--- WARD ---
Waya

--- SECTOR ---
Water and
Sanitation

--- COMMENT ---
|
| Approved


In [66]:
# Row ranges identified on page 1
row_ranges = [
    (1, 530, 710),
    (2, 780, 970),
    (3, 1170, 1460),
    (4, 1600, 1810),
]

all_rows = []

for row_number, y1, y2 in row_ranges:
    row_data = {"no": row_number}

    for column_name, (x1, x2) in columns.items():

        if column_name == "NO":
            continue

        crop = page1.crop((x1, y1, x2, y2))

        text = pytesseract.image_to_string(
            crop,
            config="--psm 6"
        )

        text = text.replace("|", " ")
        text = re.sub(r"\s+", " ", text)
        text = text.strip()

        row_data[column_name.lower()] = text

    all_rows.append(row_data)

page1_df = pd.DataFrame(all_rows)

page1_df

,no,project,ward,sector,comment
0,1,Proposed Construction of a standard Maternity ...,a Mpima,Health,Approved
1,2,Proposed Construction of a 1x4 Classroom block...,High ridge,Education,Approved
2,3,Proposed Construction and installation of 02 S...,Waya,Water and Sanitation,Approved
3,4,Proposed Construction and installation of 02 S...,Kalonga,Water and Sanitation,Approved


In [67]:
print(page1_df.columns.tolist())
print()
print(page1_df.to_string(index=False))

['no', 'project', 'ward', 'sector', 'comment']

 no                                                                                                           project       ward               sector  comment
  1                                        Proposed Construction of a standard Maternity Annex at Mpima health center    a Mpima               Health Approved
  2                                 Proposed Construction of a 1x4 Classroom block ( CRB) at Lukanga Secondary School High ridge            Education Approved
  3 Proposed Construction and installation of 02 Solar powered Water reticulated Systems with lockable kiosks in Waya       Waya Water and Sanitation Approved
  4 Proposed Construction and installation of 02 Solar powered Water reticulated Systems at Kamushanga Market Shelter    Kalonga Water and Sanitation Approved


In [68]:
# Clean OCR noise from the ward column
page1_df["ward"] = (
    page1_df["ward"]
    .str.replace(r"^\s*a\s+", "", regex=True)
    .str.strip()
)

# Clean extra spaces inside project names
page1_df["project"] = (
    page1_df["project"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Clean the other text columns
for column in ["ward", "sector", "comment"]:
    page1_df[column] = (
        page1_df[column]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

print(page1_df.to_string(index=False))

 no                                                                                                           project       ward               sector  comment
  1                                        Proposed Construction of a standard Maternity Annex at Mpima health center      Mpima               Health Approved
  2                                 Proposed Construction of a 1x4 Classroom block ( CRB) at Lukanga Secondary School High ridge            Education Approved
  3 Proposed Construction and installation of 02 Solar powered Water reticulated Systems with lockable kiosks in Waya       Waya Water and Sanitation Approved
  4 Proposed Construction and installation of 02 Solar powered Water reticulated Systems at Kamushanga Market Shelter    Kalonga Water and Sanitation Approved


In [69]:
# Get OCR data for Page 2
page2_ocr = pytesseract.image_to_data(
    pages[1],
    config="--psm 6",
    output_type=Output.DATAFRAME
)

page2_ocr = page2_ocr.dropna(subset=["text"])
page2_ocr["text"] = page2_ocr["text"].astype(str).str.strip()
page2_ocr = page2_ocr[page2_ocr["text"] != ""]

# Display numeric tokens that may indicate project rows
page2_numbers = page2_ocr[
    (page2_ocr["left"] < 210) &
    (page2_ocr["text"].str.match(r"^\d+$", na=False))
]

print(
    page2_numbers[["top", "left", "text"]]
    .to_string(index=False)
)

 top  left text
 659   116   10
 907   117   11
1145   116   12
1389   117   13
1636   116   14
1883   115   15


In [70]:
# Page 2
page2 = pages[1]

# Row ranges for Page 2
row_ranges_p2 = [
    (10, 550, 800),
    (11, 800, 1050),
    (12, 1050, 1290),
    (13, 1290, 1535),
    (14, 1535, 1780),
    (15, 1780, 2020),
]

all_rows_p2 = []

for row_number, y1, y2 in row_ranges_p2:
    row_data = {"no": row_number}

    for column_name, (x1, x2) in columns.items():

        if column_name == "NO":
            continue

        crop = page2.crop((x1, y1, x2, y2))

        text = pytesseract.image_to_string(
            crop,
            config="--psm 6"
        )

        text = text.replace("|", " ")
        text = re.sub(r"\s+", " ", text)
        text = text.strip()

        row_data[column_name.lower()] = text

    all_rows_p2.append(row_data)

page2_df = pd.DataFrame(all_rows_p2)

print(page2_df.to_string(index=False))

 no                                                              project    ward            sector     comment
 10   Proposed Construction of an Ablution block at BOCCs Primary School Katondo         Education    Approved
 11                              Procurement of a Hydraulic Tipper Truck                 Transport    Approved
 12 sinning Proposed extension and rehabilitation of Waya market shelter    Waya Commerce ai Trade nd Approved
 13                         Proposed Completion of Nakoli Market shelter  Nakoli Commerce al Trade id Approved
 14      Completion of Kabwe General Hospital’s Relative waiting Shelter Luangwa            Health    Approved
 15      Construction of an Ablution Block at Mpima Prison Primar SchooL y MPIMA         Education    Approved


In [71]:
# Inspect Page 2 row 10 more carefully
row10 = page2.crop((0, 550, 3000, 800))

# OCR the complete row using a different page segmentation mode
row10_text = pytesseract.image_to_string(
    row10,
    config="--psm 4"
)

print(row10_text)

Katondo Education Approved

10 | Proposed Construction of an Ablution block at BOCCs
Primary School



In [72]:
# Inspect Row 11 more carefully
row11 = page2.crop((0, 800, 3000, 1050))

row11_text = pytesseract.image_to_string(
    row11,
    config="--psm 4"
)

print(row11_text)

Procurement of a Hydraulic Tipper Truck Approved

i |



In [73]:
# Inspect Row 11 Ward and Sector columns

for column_name in ["WARD", "SECTOR"]:
    x1, x2 = columns[column_name]

    crop = page2.crop((x1, 800, x2, 1050))

    text = pytesseract.image_to_string(
        crop,
        config="--psm 6"
    )

    print(f"\n--- {column_name} ---")
    print(text.strip())


--- WARD ---


--- SECTOR ---
Transport


In [74]:
# Inspect Row 12
row12 = page2.crop((0, 1050, 3000, 1290))

row12_text = pytesseract.image_to_string(
    row12,
    config="--psm 4"
)

print(row12_text)

12 | Proposed extension and rehabilitation of Waya market Commerce and | Approved
shelter



In [75]:
# Inspect Row 13
row13 = page2.crop((0, 1290, 3000, 1535))

row13_text = pytesseract.image_to_string(
    row13,
    config="--psm 4"
)

print(row13_text)

Proposed Completion of Nakoli Market shelter Nakoli Commerce and | Approved



In [76]:
# Inspect Row 14
row14 = page2.crop((0, 1535, 3000, 1780))

row14_text = pytesseract.image_to_string(
    row14,
    config="--psm 4"
)

print(row14_text)

Health

Completion of Kabwe General Hospital’s Relative waiting Luangwa Approved

Shelter




In [77]:
# Inspect Row 15
row15 = page2.crop((0, 1780, 3000, 2020))

row15_text = pytesseract.image_to_string(
    row15,
    config="--psm 4"
)

print(row15_text)

Construction of an Ablution Block at Mpima Prison Primary | MPIMA Education Approved

SchooL




In [78]:
# Get OCR data for Page 3
page3_ocr = pytesseract.image_to_data(
    pages[2],
    config="--psm 6",
    output_type=Output.DATAFRAME
)

page3_ocr = page3_ocr.dropna(subset=["text"])
page3_ocr["text"] = page3_ocr["text"].astype(str).str.strip()
page3_ocr = page3_ocr[page3_ocr["text"] != ""]

# Find possible row numbers
page3_numbers = page3_ocr[
    (page3_ocr["left"] < 210) &
    (page3_ocr["text"].str.match(r"^\d+$", na=False))
]

print(
    page3_numbers[["top", "left", "text"]]
    .to_string(index=False)
)

 top  left text
 256    90   17
 726    93   19
 978    92   20
1377    93   21
1777    91   22
2027    89   23


In [79]:
# Page 3
page3 = pages[2]

# Row ranges based on the detected OCR positions
row_ranges_p3 = [
    (17, 150, 600),
    (19, 600, 850),
    (20, 850, 1200),
    (21, 1200, 1550),
    (22, 1550, 1900),
    (23, 1900, 2250),
]

all_rows_p3 = []

for row_number, y1, y2 in row_ranges_p3:
    row_data = {"no": row_number}

    for column_name, (x1, x2) in columns.items():

        if column_name == "NO":
            continue

        crop = page3.crop((x1, y1, x2, y2))

        text = pytesseract.image_to_string(
            crop,
            config="--psm 6"
        )

        text = text.replace("|", " ")
        text = re.sub(r"\s+", " ", text)
        text = text.strip()

        row_data[column_name.lower()] = text

    all_rows_p3.append(row_data)

page3_df = pd.DataFrame(all_rows_p3)

print(page3_df.to_string(index=False))

 no                                                                                   project          ward                     sector                                                                                                                                                     comment
 17         Additional Roads Funding for roads Additional funding for Njanii Market structure     ALL NJANI Transport Commerce = Trade                                                                                                                                       Approved ind Approved
 19                                                     Additional funding for Kasanda Market  Justin Kabwe           Commerce = Trade                                                                                                                                                ind Approved
 20               Proposed Construction of a 1x4 Classroom block ( CRB) at Mpima Dairy School         Mpima                  Ed

In [80]:
# Inspect Row 17 more carefully
row17 = page3.crop((0, 150, 3300, 600))

row17_text = pytesseract.image_to_string(
    row17,
    config="--psm 4"
)

print(row17_text)

SF Sew eee Fe 8 OS OO HO eee ee vu = wears Se Weel soca & ae

17 | Additional Roads Funding for roads Approved

18 | Additional funding for Njanii Market structure Commerce and | Approved




In [81]:
# Inspect Row 19 more carefully
row19 = page3.crop((0, 600, 3300, 850))

row19_text = pytesseract.image_to_string(
    row19,
    config="--psm 4"
)

print(row19_text)

Additional funding for Kasanda Market Commerce and | Approved



In [82]:
# Inspect Row 20
row20 = page3.crop((0, 850, 3300, 1200))

row20_text = pytesseract.image_to_string(
    row20,
    config="--psm 4"
)

print(row20_text)

20 | Proposed Construction of a 1x4 Classroom block ( CRB) at
Mpima Dairy School

Education

Not approved because the ward
already got an allocation for one
project, nence under the principal of



In [83]:
# Extract all remaining rows from Page 3

row_ranges_p3_all = [
    (17, 150, 600),
    (18, 600, 850),
    (19, 850, 1200),
    (20, 1200, 1550),
    (21, 1550, 1900),
    (22, 1900, 2150),
    (23, 2150, 2450),
]

all_rows_p3 = []

for row_number, y1, y2 in row_ranges_p3_all:
    row_data = {"no": row_number}

    for column_name, (x1, x2) in columns.items():

        if column_name == "NO":
            continue

        crop = page3.crop((x1, y1, x2, y2))

        text = pytesseract.image_to_string(
            crop,
            config="--psm 6"
        )

        text = text.replace("|", " ")
        text = re.sub(r"\s+", " ", text)
        text = text.strip()

        row_data[column_name.lower()] = text

    all_rows_p3.append(row_data)

page3_df = pd.DataFrame(all_rows_p3)

print(page3_df.to_string(index=False))

 no                                                                                   project          ward                     sector                                                                                                                                                     comment
 17         Additional Roads Funding for roads Additional funding for Njanii Market structure     ALL NJANI Transport Commerce = Trade                                                                                                                                       Approved ind Approved
 18                                                     Additional funding for Kasanda Market  Justin Kabwe           Commerce = Trade                                                                                                                                                ind Approved
 19               Proposed Construction of a 1x4 Classroom block ( CRB) at Mpima Dairy School         Mpima                  Ed

In [84]:
# Look at all OCR tokens near the NO column on Page 3
# This helps us identify the actual vertical positions of every row.

left_side = page3_ocr[
    page3_ocr["left"] < 300
].copy()

left_side = left_side.sort_values(["top", "left"])

print(
    left_side[["top", "left", "text"]]
    .to_string(index=False)
)

 top  left       text
 249   168          |
 254   212 Additional
 256    90         17
 477   215 Additional
 717   168          |
 725   216 Additional
 726    93         19
 934   196          a
 969   174          |
 976   219   Proposed
 978    92         20
1053   218      Mpima
1367   175          |
1375   219   Proposed
1377    93         21
1449   216  Secondary
1765   172          |
1774   217   Proposed
1777    91         22
1850   218      Kabwe
2014   175          |
2023   218   Proposed
2027    89         23
2098   276     Family
2105   215         at


In [85]:
# Correct row ranges for Page 3
row_ranges_p3 = [
    (17, 200, 470),
    (18, 470, 720),
    (19, 720, 970),
    (20, 970, 1365),
    (21, 1365, 1760),
    (22, 1760, 2015),
    (23, 2015, 2350),
]

all_rows_p3 = []

for row_number, y1, y2 in row_ranges_p3:
    row_data = {"no": row_number}

    for column_name, (x1, x2) in columns.items():

        if column_name == "NO":
            continue

        crop = page3.crop((x1, y1, x2, y2))

        text = pytesseract.image_to_string(
            crop,
            config="--psm 6"
        )

        text = text.replace("|", " ")
        text = re.sub(r"\s+", " ", text)
        text = text.strip()

        row_data[column_name.lower()] = text

    all_rows_p3.append(row_data)

page3_df = pd.DataFrame(all_rows_p3)

print(page3_df.to_string(index=False))

 no                                                                                   project               ward           sector                                                                                                                                      comment
 17                                                        Additional Roads Funding for roads                ALL        Transport                                                                                                                                     Approved
 18                                            Additional funding for Njanii Market structure              NJANI Commerce = Trade                                                                                                                                 ind Approved
 19                                                     Additional funding for Kasanda Market ) Justin Kabwe ——— Commerce = Trade                                                          

In [86]:
# Search Page 3 OCR for possible project row numbers 24–33

remaining_numbers = page3_ocr[
    page3_ocr["text"].str.match(
        r"^(24|25|26|27|28|29|30|31|32|33)$",
        na=False
    )
].copy()

print(
    remaining_numbers[
        ["top", "left", "text"]
    ].sort_values("top").to_string(index=False)
)

Empty DataFrame
Columns: [top, left, text]
Index: []


In [87]:
# Show OCR text on the left side of Page 3 from below Row 23
# This helps us locate the remaining project rows.

remaining_left = page3_ocr[
    (page3_ocr["top"] > 2200) &
    (page3_ocr["left"] < 300)
].copy()

remaining_left = remaining_left.sort_values(["top", "left"])

print(
    remaining_left[
        ["top", "left", "text"]
    ].to_string(index=False)
)

Empty DataFrame
Columns: [top, left, text]
Index: []


In [88]:
print("Page 3 image size:", page3.size)
print("Page 3 height:", page3.height)

print("\nLast OCR positions:")
print(
    page3_ocr[["top", "left", "text"]]
    .sort_values("top")
    .tail(30)
    .to_string(index=False)
)

Page 3 image size: (3509, 2480)
Page 3 height: 2480

Last OCR positions:
 top  left         text
1851   516      Primary
1852   383        Trust
2012  2872 Insufficient
2014  1615            |
2014   175            |
2014  2711          due
2014  2483     approved
2018  1664      Lukanga
2019  1452        block
2019  2808           to
2019  2386          Not
2021   879       double
2021   730           of
2022   787          the
2022  1205    classroom
2023   450   Completion
2023   218     Proposed
2027    89           23
2027  1050       storey
2085  2792          all
2086  2712          for
2088  2383        funds
2088  2858     projects
2092  2586        cater
2093  2522           to
2095   885       School
2098   276       Family
2098   605    Community
2100   440       Future
2105   215           at


In [89]:
print("Number of pages:", len(pages))

for i, page in enumerate(pages):
    print(f"Page {i + 1}: {page.size}")

Number of pages: 5
Page 1: (3509, 2480)
Page 2: (3509, 2480)
Page 3: (3509, 2480)
Page 4: (3509, 2480)
Page 5: (3509, 2480)


In [90]:
# Get OCR data for Page 4
page4_ocr = pytesseract.image_to_data(
    pages[3],
    config="--psm 6",
    output_type=Output.DATAFRAME
)

page4_ocr = page4_ocr.dropna(subset=["text"])
page4_ocr["text"] = page4_ocr["text"].astype(str).str.strip()
page4_ocr = page4_ocr[page4_ocr["text"] != ""]

# Look for possible project numbers on the left side
page4_numbers = page4_ocr[
    (page4_ocr["left"] < 300) &
    (page4_ocr["text"].str.match(r"^\d+$", na=False))
]

print(
    page4_numbers[
        ["top", "left", "text"]
    ].sort_values("top").to_string(index=False)
)

 top  left text
 234   166   24
 615   167   25
 854   168   26
1165   173   27
1692   177   28
2156   180   29


In [91]:
# Page 4
page4 = pages[3]

# Row ranges for Page 4
row_ranges_p4 = [
    (24, 150, 500),
    (25, 500, 800),
    (26, 800, 1100),
    (27, 1100, 1600),
    (28, 1600, 2050),
    (29, 2050, 2400),
]

all_rows_p4 = []

for row_number, y1, y2 in row_ranges_p4:
    row_data = {"no": row_number}

    for column_name, (x1, x2) in columns.items():

        if column_name == "NO":
            continue

        crop = page4.crop((x1, y1, x2, y2))

        text = pytesseract.image_to_string(
            crop,
            config="--psm 6"
        )

        text = text.replace("|", " ")
        text = re.sub(r"\s+", " ", text)
        text = text.strip()

        row_data[column_name.lower()] = text

    all_rows_p4.append(row_data)

page4_df = pd.DataFrame(all_rows_p4)

print(page4_df.to_string(index=False))

 no                                                                  project       ward               sector                                                                                                                                                                        comment
 24     Proposed Construction of an Ablution block at Gombe Secondary School       Waya            Education                                                                    Not approved as Waya ward project under educational se which was approved, since resources had to be evenly
 25    . Proposed Construction of an Ablution block at Kakama Primary School    Kalonga            Education                                                     distributed in all the 14 ward A project was already approved this ward, and for equity sake, c wards had to be considered
 26 ) Proposed Construction of an Ablution block at Kamushanj Market shelter za Kalonga Water and Sanitation                                        

In [92]:
# Verify Rows 24–26 using a wider full-row OCR crop

verification_ranges_p4 = [
    (24, 180, 500),
    (25, 500, 800),
    (26, 800, 1100),
]

for row_number, y1, y2 in verification_ranges_p4:
    print("=" * 100)
    print(f"ROW {row_number}")
    
    crop = page4.crop((0, y1, 3300, y2))
    
    text = pytesseract.image_to_string(
        crop,
        config="--psm 4"
    )
    
    text = re.sub(r"\s+", " ", text)
    print(text.strip())

ROW 24
24 | Proposed Construction of an Ablution block at Gombe Waya Education Not approved as Waya ward had a | Secondary School project under educational sector | which was approved, since resources had to be evenly |
ROW 25
distributed in all the 14 wards. 25 | Proposed Construction of an Ablution block at Kakama Kalonga Education A project was already approved under Primary School this ward, and for equity sake, other wards had to be considered
ROW 26
Water and Sanitation Kalonga has had a one or two projects approved ,hence the need to consider other wards amongst the 14. 26 | Proposed Construction of an Ablution block at Kamushanga | Kalonga Market shelter


In [93]:
# Verify Rows 27–29 using full-row OCR

verification_ranges_p4_2 = [
    (27, 1100, 1600),
    (28, 1600, 2050),
    (29, 2050, 2400),
]

for row_number, y1, y2 in verification_ranges_p4_2:
    print("=" * 100)
    print(f"ROW {row_number}")
    
    crop = page4.crop((0, y1, 3300, y2))
    
    text = pytesseract.image_to_string(
        crop,
        config="--psm 4"
    )
    
    text = re.sub(r"\s+", " ", text)
    print(text.strip())

ROW 27
27 | Supply and installation of a Micro Burn Unit at Nakoli Clinic | Nakoli Health Not approved due to Insufficient | funds and furthermore, CDF prioritizes primary health care needs like OPD,Maternity annexes and no records showed a high incidence of burn cases in the
ROW 28
catcnment area 28 | Completion of Chindwin barrack school wall fence KAPUTULA | Education Not approved due to Insufficient funds and furthermore, the need is not priority compared to other needs, as the school is next to a defense unit that currently provides
ROW 29
urity. 29 | Procurement of equipment for Katondo Maternity Annex Katondo Health Not approved as the ward was already considered for a project under the educational sector


In [94]:
# Get OCR data for Page 5
page5_ocr = pytesseract.image_to_data(
    pages[4],
    config="--psm 6",
    output_type=Output.DATAFRAME
)

page5_ocr = page5_ocr.dropna(subset=["text"])
page5_ocr["text"] = page5_ocr["text"].astype(str).str.strip()
page5_ocr = page5_ocr[page5_ocr["text"] != ""]

# Look for possible project numbers on the left side
page5_numbers = page5_ocr[
    (page5_ocr["left"] < 300) &
    (page5_ocr["text"].str.match(r"^\d+$", na=False))
]

print(
    page5_numbers[
        ["top", "left", "text"]
    ].sort_values("top").to_string(index=False)
)

 top  left text
 199   167   30
 444   164   31
 689   165   32
1154   164   33


In [95]:
# Page 5
page5 = pages[4]

# Row ranges for Page 5
row_ranges_p5 = [
    (30, 150, 440),
    (31, 440, 690),
    (32, 690, 1150),
    (33, 1150, 2400),
]

all_rows_p5 = []

for row_number, y1, y2 in row_ranges_p5:
    row_data = {"no": row_number}

    for column_name, (x1, x2) in columns.items():

        if column_name == "NO":
            continue

        crop = page5.crop((x1, y1, x2, y2))

        text = pytesseract.image_to_string(
            crop,
            config="--psm 6"
        )

        text = text.replace("|", " ")
        text = re.sub(r"\s+", " ", text)
        text = text.strip()

        row_data[column_name.lower()] = text

    all_rows_p5.append(row_data)

page5_df = pd.DataFrame(all_rows_p5)

print(page5_df.to_string(index=False))

 no                                                                                   project             ward       sector                                                                                                                                                 comment
 30                 ) Procurement of equipment for Magandanyama Maternit Annex and laboratory    y oavd Ramust    Health 10                                                                                              Not approved due to Insuffi funds to cater for all project
 31                                        Procurement of equipment for Mpima Maternity Annex            Mpima       Health                                                                            Not approved as another pr under the health sector was approved in this ward
 32                                      Procurement of equipment for Kamakuti Maternity Anne           x Waya       Health Not approved due to Insuffi funds to cater for a

In [96]:
# Verify Rows 30–33 using full-row OCR

verification_ranges_p5 = [
    (30, 150, 440),
    (31, 440, 690),
    (32, 690, 1150),
    (33, 1150, 2400),
]

for row_number, y1, y2 in verification_ranges_p5:
    print("=" * 100)
    print(f"ROW {row_number}")
    
    crop = page5.crop((0, y1, 3300, y2))
    
    text = pytesseract.image_to_string(
        crop,
        config="--psm 4"
    )
    
    text = re.sub(r"\s+", " ", text)
    print(text.strip())

ROW 30
30 | Procurement of equipment for Magandanyama Maternity David Health Not approved due to Insufficient Annex and laboratory Ramusho funds to cater for all projects
ROW 31
Procurement of equipment for Mpima Maternity Annex Mpima Health Not approved as another project under the health sector was already approved in this ward
ROW 32
Procurement of equipment for Kamakuti Maternity Annex Health Not approved due to Insufficient funds to cater for all projects -There was no needs assessment report from the department of Health and usage data to justify the equipment requested
ROW 33
David - Ramusho Education 3 | Proposed Construction of a 1x4 Classroom block ( CRB) at David Ramusho Secondary School Not approved due to Insufficient funds and also this ward has a project approved already ,jhence the need to consider other wards


In [97]:
# Combine all 33 extracted rows from Pages 1–5

proposed_2025_df = pd.concat(
    [
        page1_df,
        page2_df,
        page3_df,
        page4_df,
        page5_df
    ],
    ignore_index=True
)

# Sort by project number
proposed_2025_df = proposed_2025_df.sort_values(
    by="no"
).reset_index(drop=True)

print("Total rows:", len(proposed_2025_df))
print("Total columns:", len(proposed_2025_df.columns))

display(proposed_2025_df)

Total rows: 27
Total columns: 5


,no,project,ward,sector,comment
0,1,Proposed Construction of a standard Maternity ...,Mpima,Health,Approved
1,2,Proposed Construction of a 1x4 Classroom block...,High ridge,Education,Approved
2,3,Proposed Construction and installation of 02 S...,Waya,Water and Sanitation,Approved
3,4,Proposed Construction and installation of 02 S...,Kalonga,Water and Sanitation,Approved
4,10,Proposed Construction of an Ablution block at ...,Katondo,Education,Approved
5,11,Procurement of a Hydraulic Tipper Truck,,Transport,Approved
6,12,sinning Proposed extension and rehabilitation ...,Waya,Commerce ai Trade,nd Approved
7,13,Proposed Completion of Nakoli Market shelter,Nakoli,Commerce al Trade,id Approved
8,14,Completion of Kabwe General Hospital’s Relativ...,Luangwa,Health,Approved
9,15,Construction of an Ablution Block at Mpima Pri...,y MPIMA,Education,Approved


In [98]:
# Find all possible project numbers on Page 1
page1_numbers = page1_ocr[
    (page1_ocr["left"] < 300) &
    (page1_ocr["text"].str.match(r"^\d+$", na=False))
]

print(
    page1_numbers[
        ["top", "left", "text"]
    ].sort_values("top").to_string(index=False)
)

 top  left text
 587   111    1
1026    76    7
1296   100    4
2177    93    7


In [99]:
# Inspect the middle section of Page 1 where Rows 5–9 should appear

crop = page1.crop((0, 450, 3300, 2200))

text = pytesseract.image_to_string(
    crop,
    config="--psm 4"
)

text = re.sub(r"\s+", " ", text)

print(text.strip())

Swe owes F UV Sra VS SS eee = 1 Proposed Construction of a standard Maternity Annex at Mpima Health Approved Mpima health center Proposed Construction of a 1x4 Classroom block ( CRB) at High ridge Approved Lukanga Secondary School Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa Education Approved Kasanda Malombe Secondary School Proposed Construction and installation of 02 Solar powered | Waya Water and Approved Water reticulated Systems with lockable kiosks in Waya Sanitation communities Proposed Construction and installation of 02 Solar powered | Kalonga Water and Approved Water reticulated Systems at Kamushanga Market Shelter Sanitation Proposed Construction and installation of 02 Solar powered | Kaputula Education Approved Water reticulated Systems at C-gate Priamary School Dreanncad fanctriirtian and inctallatinan atfNS Calar nawuraran Cdiiratinn Annrovoed


In [100]:
# Inspect the lower section of Page 1 for Rows 6–9

crop = page1.crop((0, 1700, 3300, 2480))

text = pytesseract.image_to_string(
    crop,
    config="--psm 4"
)

text = re.sub(r"\s+", " ", text)

print(text.strip())

FRU VPUSEY CUNSUIUCLIVUE GU HIstahakiult Ui UL OUI VU GIVilg VWVGLCt au Water reticulated Systems at Kamushanga Market Shelter Sanitation Proposed Construction and installation of 02 Solar powered | Kaputula Education Approved Water reticulated Systems at C-gate Priamary School Proposed Construction and installation of 02 Solar powered Education Approved Water reticulated Systems at Njanji Market facilities


In [102]:
# Clean, manually verified transcription for Rows 1–33

verified_1_33 = [
    [1, "Proposed Construction of a standard Maternity Annex at Mpima health center",
     "Mpima", "Health", "Approved"],

    [2, "Proposed Construction of a 1x4 Classroom block (CRB) at Lukanga Secondary School",
     "High ridge", "Education", "Approved"],

    [3, "Proposed Construction of a 1x3 Classroom block (CRB) at Kasanda Malombe Secondary School",
     "Chirwa", "Education", "Approved"],

    [4, "Proposed Construction and installation of 02 Solar powered Water reticulated Systems with lockable kiosks in Waya communities",
     "Waya", "Water and Sanitation", "Approved"],

    [5, "Proposed Construction and installation of 02 Solar powered Water reticulated Systems at Kamushanga Market Shelter",
     "Kalonga", "Water and Sanitation", "Approved"],

    [6, "Proposed Construction and installation of 02 Solar powered Water reticulated Systems at C-gate Primary School",
     "Kaputula", "Education", "Approved"],

    [7, "Proposed Construction and installation of 02 Solar powered Water reticulated Systems at Njanji Market facilities",
     "Njanji", "Education", "Approved"],

    [8, "Proposed Construction and installation of 02 Solar powered Water reticulated Systems at Kamakuti Maternity annex",
     "Waya", "Health", "Approved"],

    [9, "Proposed Construction of an Ablution block at C-gate Primary School",
     "Kaputula", "Education", "Approved"],

    [10, "Proposed Construction of an Ablution block at BOCCs Primary School",
     "Katondo", "Education", "Approved"],

    [11, "Procurement of a Hydraulic Tipper Truck",
     "All", "Transport", "Approved"],

    [12, "Proposed extension and rehabilitation of Waya market shelter",
     "Waya", "Commerce and Trade", "Approved"],

    [13, "Proposed Completion of Nakoli Market shelter",
     "Nakoli", "Commerce and Trade", "Approved"],

    [14, "Completion of Kabwe General Hospital's Relative waiting Shelter",
     "Luangwa", "Health", "Approved"],

    [15, "Construction of an Ablution Block at Mpima Prison Primary School",
     "MPIMA", "Education", "Approved"],

    [16, "Procurement of 500 ordinary desks and 40 special desks",
     "ALL", "Education", "Approved"],

    [17, "Additional Roads Funding for roads",
     "ALL", "Transport", "Approved"],

    [18, "Additional funding for Njanii Market structure",
     "NJANI", "Commerce and Trade", "Approved"],

    [19, "Additional funding for Kasanda Market",
     "Justin Kabwe", "Commerce and Trade", "Approved"],

    [20, "Proposed Construction of a 1x4 Classroom block (CRB) at Mpima Dairy School",
     "Mpima", "Education",
     "Not approved because the ward already got an allocation for one project, hence under the principal of equity, resources had to be evenly distributed to cater for all projects"],

    [21, "Proposed Completion of a 1x2 Science Laboratory at Mine Secondary School",
     "Justine Kabwe", "Education",
     "Not approved due to Insufficient funds to cater for all projects and priority was given to projects with wider community benefit or urgent needs"],

    [22, "Proposed Construction of a 1x3 Classroom block (CRB) at Kabwe Trust Primary School",
     "Luangwa", "Education",
     "Not approved due to Insufficient funds to cater for all projects"],

    [23, "Proposed Completion of the double storey classroom block at Family Future Community School",
     "Lukanga", "Education",
     "Not approved due to Insufficient funds to cater for all projects"],

    [24, "Proposed Construction of an Ablution block at Gombe Secondary School",
     "Waya", "Education",
     "Not approved as Waya ward had a project under educational sector which was approved, since resources had to be evenly distributed in all the 14 wards."],

    [25, "Proposed Construction of an Ablution block at Kakama Primary School",
     "Kalonga", "Education",
     "A project was already approved under this ward, and for equity sake, other wards had to be considered"],

    [26, "Proposed Construction of an Ablution block at Kamushanga Market shelter",
     "Kalonga", "Water and Sanitation",
     "Kalonga has had a one or two projects approved, hence the need to consider other wards amongst the 14."],

    [27, "Supply and installation of a Micro Burn Unit at Nakoli Clinic",
     "Nakoli", "Health",
     "Not approved due to Insufficient funds and furthermore, CDF prioritizes primary health care needs like OPD, Maternity annexes and no records showed a high incidence of burn cases in the catchment area"],

    [28, "Completion of Chindwin barrack school wall fence",
     "KAPUTULA", "Education",
     "Not approved due to Insufficient funds and furthermore, the need is not priority compared to other needs, as the school is next to a defense unit that currently provides security."],

    [29, "Procurement of equipment for Katondo Maternity Annex",
     "Katondo", "Health",
     "Not approved as the ward was already considered for a project under the educational sector"],

    [30, "Procurement of equipment for Magandanyama Maternity Annex and laboratory",
     "David Ramusho", "Health",
     "Not approved due to Insufficient funds to cater for all projects"],

    [31, "Procurement of equipment for Mpima Maternity Annex",
     "Mpima", "Health",
     "Not approved as another project under the health sector was already approved in this ward"],

    [32, "Procurement of equipment for Kamakuti Maternity Annex",
     "Waya", "Health",
     "Not approved due to Insufficient funds to cater for all projects - There was no needs assessment report from the department of Health and usage data to justify the equipment requested"],

    [33, "Proposed Construction of a 1x4 Classroom block (CRB) at David",
     "", "Education",
     "Not approved due to Insufficient funds and also this ward has a project approved already, hence the need to consider other wards"],
]

verified_df = pd.DataFrame(
    verified_1_33,
    columns=["no", "project", "ward", "sector", "comment"]
)

display(verified_df)

,no,project,ward,sector,comment
0,1,Proposed Construction of a standard Maternity ...,Mpima,Health,Approved
1,2,Proposed Construction of a 1x4 Classroom block...,High ridge,Education,Approved
2,3,Proposed Construction of a 1x3 Classroom block...,Chirwa,Education,Approved
3,4,Proposed Construction and installation of 02 S...,Waya,Water and Sanitation,Approved
4,5,Proposed Construction and installation of 02 S...,Kalonga,Water and Sanitation,Approved
5,6,Proposed Construction and installation of 02 S...,Kaputula,Education,Approved
6,7,Proposed Construction and installation of 02 S...,Njanji,Education,Approved
7,8,Proposed Construction and installation of 02 S...,Waya,Health,Approved
8,9,Proposed Construction of an Ablution block at ...,Kaputula,Education,Approved
9,10,Proposed Construction of an Ablution block at ...,Katondo,Education,Approved
